# 06 Native DQA-MoE From Warmup

Goal: train DQA-routed anonymous MoE from the warmup stage instead of upcycling MoE only after FedSTO phase1.

Key differences from 05:
- Backbone, neck, and head MoE slots exist from warmup.
- Phase1 trains adapter/head MoE rather than dense backbone only.
- Server repair is constrained to BN + head MoE so it does not erase client-specialized adapter experts.
- Phase1 and phase2 use trust-region soft aggregation for adapter/head tensors.
- Aggregation is expert-wise: per-client server-val mAP, router usage, expert delta norm, and anonymous DQA/client prior decide how strongly each expert delta is mixed.
- Routing remains anonymous; DQA context and feature-quality stats bias routing, but experts are not manually named as domain experts.

In [ ]:
from pathlib import Path

REPO = Path('/app/Object_Detection')
SCRIPT = REPO / 'dynamic_quality_aware_classwise_aggregation/dqa_moe_trust_region/scripts/run_06_native_dqa_moe_from_warmup_full.sh'
WORKSPACE = REPO / 'dynamic_quality_aware_classwise_aggregation/dqa_moe_trust_region/output/06_native_dqa_moe_from_warmup_full'
SUMMARY = WORKSPACE / 'anonymous_backbone_moe_round_summary.csv'
ROUTER = WORKSPACE / 'anonymous_backbone_moe_router_diagnostics.csv'
EXPERT_AGG = WORKSPACE / 'expertwise_aggregation_diagnostics.csv'
LOG = WORKSPACE / 'native_dqa_moe_from_warmup_full.log'

SCRIPT, WORKSPACE

In [ ]:
# Full run. This is intentionally not an early-stop experiment.
!mkdir -p {WORKSPACE}
!stdbuf -oL -eL bash {SCRIPT} 2>&1 | tee -a {LOG}

In [ ]:
if EXPERT_AGG.exists():
    expert_agg = pd.read_csv(EXPERT_AGG)
    display(expert_agg[['phase', 'round', 'scope', 'mean_expert_entropy', 'min_expert_entropy', 'max_expert_weight', 'updated_tensors', 'clipped_tensors']])
else:
    print('expert-wise aggregation diagnostics do not exist yet:', EXPERT_AGG)

In [ ]:
import pandas as pd

if SUMMARY.exists():
    display(pd.read_csv(SUMMARY))
else:
    print('summary does not exist yet:', SUMMARY)

In [ ]:
if ROUTER.exists():
    router = pd.read_csv(ROUTER)
    display(router[['phase', 'round', 'split', 'images', 'mean_entropy', 'mean_max_prob', 'mean_active_experts']])
else:
    print('router diagnostics do not exist yet:', ROUTER)